**Explanation Technique Selector** </br></br>
The Explanation Technique Selector module helps users develop the skill of matching appropriate explainability methods to different audiences and contexts. Through interactive scenarios, users practice selecting from various explanation techniques (feature importance, LIME, SHAP, counterfactuals, example-based explanations, rule extractions, and natural language summaries) based on audience characteristics like technical background, decision context, and information needs. Each scenario presents a stakeholder requesting AI transparency, and users must choose the most suitable explanation approach, receiving immediate feedback on why certain techniques work better for specific audiences. This builds the critical skill of tailoring technical explanations to diverse stakeholder needs rather than using one-size-fits-all approaches.

In [ ]:
#@title pip install
from IPython.display import clear_output

!pip install ipywidgets
clear_output()  # Hides the output after the cell runs

In [ ]:
#@title Imports
import random
import time
from IPython.display import display
import ipywidgets as widgets
from ipywidgets import Button, VBox, HBox, HTML, Layout, RadioButtons

In [ ]:
#@title Explanation Techniques
explanation_techniques = {
    "Feature Importance": {
        "description": "Shows which input variables most influenced the model's decision",
        "best_for": ["technical_moderate", "business_analyst"],
        "strengths": "Simple, intuitive, computationally efficient",
        "limitations": "Doesn't show direction of influence or interactions between features",
        "example": "Bar chart showing 'Credit Score (35%), Income (25%), Employment History (20%)...'"
    },
    "LIME (Local Interpretable Model-Agnostic Explanations)": {
        "description": "Explains individual predictions by approximating the model locally with an interpretable model",
        "best_for": ["data_scientist", "technical_expert"],
        "strengths": "Model-agnostic, provides local explanations for specific predictions",
        "limitations": "Can be unstable, requires technical understanding, computationally expensive",
        "example": "'For this loan rejection: Credit Score < 650 (-0.3), High Debt Ratio (+0.2)...'"
    },
    "SHAP (SHapley Additive exPlanations)": {
        "description": "Uses game theory to assign each feature a contribution value for a prediction",
        "best_for": ["data_scientist", "technical_expert", "regulator"],
        "strengths": "Theoretically grounded, consistent, shows magnitude and direction",
        "limitations": "Computationally expensive, requires technical expertise to interpret",
        "example": "Waterfall chart showing how each feature pushed the prediction up or down from baseline"
    },
    "Counterfactual Explanations": {
        "description": "Shows what would need to change for a different outcome",
        "best_for": ["end_user", "customer_service", "business_analyst"],
        "strengths": "Actionable, intuitive, directly useful for decision-making",
        "limitations": "May not be realistic, can be misleading if changes aren't actually achievable",
        "example": "'If your credit score was 680 instead of 640, you would likely be approved'"
    },
    "Example-Based Explanations": {
        "description": "Explains by showing similar cases and their outcomes",
        "best_for": ["end_user", "executive", "customer_service"],
        "strengths": "Highly intuitive, no technical knowledge needed, builds trust",
        "limitations": "Privacy concerns, may not capture full decision logic, can reinforce biases",
        "example": "'Your application is similar to 100 other applicants, 78 were approved, 22 were denied'"
    },
    "Rule Extraction": {
        "description": "Converts complex model into interpretable if-then rules",
        "best_for": ["regulator", "compliance_officer", "business_analyst"],
        "strengths": "Clear logic, auditable, easy to verify compliance",
        "limitations": "Oversimplifies complex models, may sacrifice accuracy",
        "example": "'IF credit_score > 700 AND income > 50K AND debt_ratio < 0.4 THEN approve'"
    },
    "Natural Language Summary": {
        "description": "Generates human-readable text explaining the decision",
        "best_for": ["end_user", "executive", "customer_service", "general_public"],
        "strengths": "Accessible to everyone, can be customized to audience, feels personal",
        "limitations": "Can obscure details, may oversimplify, hard to verify accuracy",
        "example": "'Your loan was denied primarily because your credit score is below our threshold...'"
    }
}

scenarios = [
    {
        "title": "Rejected Loan Applicant",
        "context": "A customer was denied a personal loan and wants to know why and what they can do differently.",
        "audience": "end_user",
        "audience_description": "Non-technical individual seeking actionable guidance",
        "decision_stakes": "Personal financial impact",
        "best_technique": "Counterfactual Explanations",
        "why_best": "Provides actionable guidance that the customer can actually use to improve their chances. Counterfactuals are intuitive and don't require technical knowledge.",
        "runner_up": "Natural Language Summary",
        "why_not_others": "LIME/SHAP are too technical. Feature importance doesn't tell them what to do. Example-based might help but isn't as actionable."
    },
    {
        "title": "Board of Directors Quarterly Review",
        "context": "CEO needs to explain to the board how the new AI hiring system makes decisions and whether it's fair.",
        "audience": "executive",
        "audience_description": "Senior leadership with limited technical background, focused on business outcomes and risk",
        "decision_stakes": "Strategic business and legal risk",
        "best_technique": "Natural Language Summary",
        "why_best": "Executives need high-level understanding without technical details. Natural language can communicate key points about fairness and decision patterns clearly.",
        "runner_up": "Example-Based Explanations",
        "why_not_others": "LIME/SHAP/Feature Importance are too technical for board presentation. Rule extraction might work but is less digestible. Counterfactuals aren't suitable for system-wide explanation."
    },
    {
        "title": "Regulatory Audit",
        "context": "Financial regulator is auditing your credit scoring AI for compliance with fair lending laws.",
        "audience": "regulator",
        "audience_description": "Government officials needing to verify legal compliance and identify potential discrimination",
        "decision_stakes": "Legal compliance and potential fines",
        "best_technique": "SHAP (SHapley Additive exPlanations)",
        "why_best": "SHAP provides rigorous, mathematically grounded explanations that can demonstrate fair treatment across demographics. Regulators need robust, defensible analysis.",
        "runner_up": "Rule Extraction",
        "why_not_others": "Natural language and examples are too vague for compliance purposes. LIME is less stable than SHAP. Counterfactuals don't prove system-wide fairness."
    },
    {
        "title": "Data Science Team Model Review",
        "context": "Data scientists reviewing a colleague's fraud detection model to understand its decision boundaries.",
        "audience": "data_scientist",
        "audience_description": "Technical experts who understand ML algorithms and need detailed insights",
        "decision_stakes": "Model performance and technical accuracy",
        "best_technique": "SHAP (SHapley Additive exPlanations)",
        "why_best": "Data scientists can handle the complexity and benefit from SHAP's mathematical rigor and ability to reveal feature interactions and global patterns.",
        "runner_up": "LIME (Local Interpretable Model-Agnostic Explanations)",
        "why_not_others": "Natural language and examples are too superficial. Feature importance alone doesn't show interactions. Counterfactuals are more for end users."
    },
    {
        "title": "Customer Service Representative Training",
        "context": "Training customer service reps to explain to customers why the chatbot made certain product recommendations.",
        "audience": "customer_service",
        "audience_description": "Semi-technical staff who need to communicate AI decisions to customers",
        "decision_stakes": "Customer satisfaction and trust",
        "best_technique": "Example-Based Explanations",
        "why_best": "Customer service reps can easily communicate examples to customers: 'Customers with similar browsing history purchased these items.' It builds trust without technical complexity.",
        "runner_up": "Natural Language Summary",
        "why_not_others": "LIME/SHAP are too technical for customer-facing roles. Feature importance is abstract. Counterfactuals work for some cases but examples are more versatile."
    },
    {
        "title": "Hospital Ethics Committee",
        "context": "Ethics committee reviewing an AI diagnostic tool to ensure it makes medically sound and equitable decisions.",
        "audience": "compliance_officer",
        "audience_description": "Medical and ethics professionals needing to verify clinical reasoning and fairness",
        "decision_stakes": "Patient safety and ethical medical practice",
        "best_technique": "Rule Extraction",
        "why_best": "Medical ethics committees need to verify that AI reasoning aligns with medical standards. Clear if-then rules can be compared against clinical guidelines.",
        "runner_up": "SHAP (SHapley Additive exPlanations)",
        "why_not_others": "Natural language is too vague for medical verification. Examples raise privacy concerns. LIME is less stable. Feature importance doesn't show the full logic."
    },
    {
        "title": "Business Analyst Cost-Benefit Study",
        "context": "Business analyst needs to understand which factors drive AI pricing recommendations to inform pricing strategy.",
        "audience": "business_analyst",
        "audience_description": "Moderate technical skills, focused on business insights and strategy",
        "decision_stakes": "Business strategy and competitive positioning",
        "best_technique": "Feature Importance",
        "why_best": "Business analysts need to understand key drivers without getting lost in technical details. Feature importance provides clear, actionable insights for strategy.",
        "runner_up": "Counterfactual Explanations",
        "why_not_others": "LIME/SHAP are more technical than needed. Natural language might lack the quantitative rigor they need. Examples and rules don't reveal the key business drivers as clearly."
    },
    {
        "title": "Public Trust Campaign",
        "context": "Government agency launching public communication about how their AI benefit allocation system works.",
        "audience": "general_public",
        "audience_description": "Diverse public with varying technical backgrounds and high need for trust",
        "decision_stakes": "Public trust and democratic legitimacy",
        "best_technique": "Natural Language Summary",
        "why_best": "The general public needs accessible, jargon-free explanations that build trust. Natural language can be tailored to different literacy levels and cultural contexts.",
        "runner_up": "Example-Based Explanations",
        "why_not_others": "All technical methods (LIME, SHAP, Feature Importance) are too complex. Rules might work but are less engaging. Counterfactuals are more for individual cases."
    }
]

In [ ]:
#@title Explanation Technique Selector
class ExplanationTechniqueSelector:
    def __init__(self, scenarios_list, techniques_dict):
        self.scenarios = scenarios_list.copy()
        self.techniques = techniques_dict
        self.current_scenario = 0
        self.score = 0
        self.selections = []
        
        random.shuffle(self.scenarios)
        
        # Create widgets
        self.scenario_display = HTML(
            value="<h3>Explanation Technique Selector</h3>",
            layout=Layout(width='100%')
        )
        
        self.technique_selector = RadioButtons(
            options=list(self.techniques.keys()),
            description='Technique:',
            disabled=False,
            layout=Layout(width='100%')
        )
        
        self.submit_button = Button(
            description='Submit Selection',
            button_style='primary',
            layout=Layout(width='200px', height='40px', margin='10px')
        )
        
        self.feedback_display = HTML(value="")
        
        self.technique_info_display = HTML(value="")
        
        self.next_button = Button(
            description='Next Scenario →',
            button_style='',
            layout=Layout(width='150px', height='30px', margin='10px')
        )
        
        self.progress_display = HTML(value="")
        
        # Events
        self.technique_selector.observe(self.show_technique_info, names='value')
        self.submit_button.on_click(lambda b: self.submit_selection())
        self.next_button.on_click(lambda b: self.next_scenario())
        
        # Layout
        submit_box = HBox([self.submit_button], layout=Layout(justify_content='center'))
        next_box = HBox([self.next_button], layout=Layout(justify_content='center'))
        
        self.widget = VBox([
            self.scenario_display,
            self.technique_selector,
            self.technique_info_display,
            submit_box,
            self.feedback_display,
            next_box,
            self.progress_display
        ])
    
    def start_selector(self):
        self.current_scenario = 0
        self.score = 0
        self.selections = []
        self.show_scenario()
        return self.widget
    
    def show_scenario(self):
        if self.current_scenario >= len(self.scenarios):
            self.show_results()
            return
        
        scenario = self.scenarios[self.current_scenario]
        
        self.scenario_display.value = f"""
        <div style='padding: 20px; background: #f8f9fa; border-radius: 10px; margin: 15px 0;'>
            <h3 style='color: #2c3e50; margin-bottom: 15px;'>Scenario {self.current_scenario + 1}: {scenario['title']}</h3>
            <div style='background: white; padding: 15px; border-radius: 8px; margin-bottom: 15px;'>
                <p style='margin: 0 0 10px 0;'><strong>Context:</strong> {scenario['context']}</p>
                <p style='margin: 0 0 10px 0;'><strong>Audience:</strong> {scenario['audience_description']}</p>
                <p style='margin: 0;'><strong>Stakes:</strong> {scenario['decision_stakes']}</p>
            </div>
            <div style='background: #fff3cd; padding: 12px; border-radius: 8px;'>
                <p style='color: #856404; margin: 0; font-weight: bold;'>💡 Select the most appropriate explanation technique for this audience and context.</p>
            </div>
        </div>
        """
        
        # Reset selections
        self.technique_selector.value = list(self.techniques.keys())[0]
        self.feedback_display.value = ""
        self.next_button.layout.display = 'none'
        self.submit_button.disabled = False
        self.technique_selector.disabled = False
        
        self.show_technique_info({'new': self.technique_selector.value})
        self.update_progress()
    
    def show_technique_info(self, change):
        technique = change['new']
        if technique in self.techniques:
            info = self.techniques[technique]
            
            self.technique_info_display.value = f"""
            <div style='border: 2px solid #3498db; padding: 15px; border-radius: 8px; margin: 10px 0; background: #ebf3fd;'>
                <h4 style='color: #2980b9; margin: 0 0 10px 0;'>{technique}</h4>
                <p style='margin: 0 0 8px 0;'><strong>Description:</strong> {info['description']}</p>
                <p style='margin: 0 0 8px 0;'><strong>Strengths:</strong> {info['strengths']}</p>
                <p style='margin: 0 0 8px 0;'><strong>Limitations:</strong> {info['limitations']}</p>
                <p style='margin: 0; font-style: italic; color: #666;'><strong>Example:</strong> {info['example']}</p>
            </div>
            """
    
    def submit_selection(self):
        scenario = self.scenarios[self.current_scenario]
        selected = self.technique_selector.value
        best = scenario['best_technique']
        runner_up = scenario['runner_up']
        
        is_best = selected == best
        is_runner_up = selected == runner_up
        
        if is_best:
            self.score += 2
            color = "#27ae60"
            result = "✓ Excellent Choice!"
            feedback = f"<p><strong>Why this works best:</strong> {scenario['why_best']}</p>"
        elif is_runner_up:
            self.score += 1
            color = "#f39c12"
            result = "~ Good Alternative"
            feedback = f"""
            <p><strong>This could work, but there's a better option:</strong> {best}</p>
            <p><strong>Why {best} is better:</strong> {scenario['why_best']}</p>
            """
        else:
            color = "#e74c3c"
            result = "✗ Not the Best Fit"
            feedback = f"""
            <p><strong>Better choice:</strong> {best}</p>
            <p><strong>Why:</strong> {scenario['why_best']}</p>
            <p><strong>Why not others:</strong> {scenario['why_not_others']}</p>
            """
        
        self.selections.append({
            'scenario': scenario['title'],
            'selected': selected,
            'best': best,
            'correct': is_best
        })
        
        self.feedback_display.value = f"""
        <div style='border: 2px solid {color}; padding: 20px; border-radius: 10px; margin: 15px 0; background: {color}15;'>
            <h4 style='color: {color}; margin: 0 0 15px 0;'>{result}</h4>
            {feedback}
        </div>
        """
        
        self.submit_button.disabled = True
        self.technique_selector.disabled = True
        self.next_button.layout.display = 'inline-block'
        self.update_progress()
    
    def next_scenario(self):
        self.current_scenario += 1
        self.show_scenario()
    
    def update_progress(self):
        total = len(self.scenarios)
        completed = len(self.selections)
        max_score = completed * 2
        accuracy = (self.score / max_score * 100) if max_score > 0 else 0
        
        self.progress_display.value = f"""
        <div style='background: #e8f4fd; padding: 12px; border-radius: 8px; text-align: center; margin-top: 15px;'>
            <strong>Progress:</strong> {completed}/{total} scenarios |
            <strong>Score:</strong> {self.score}/{max_score} points |
            <strong>Accuracy:</strong> {accuracy:.0f}%
        </div>
        """
    
    def show_results(self):
        total = len(self.scenarios)
        max_score = total * 2
        accuracy = (self.score / max_score * 100)
        
        perfect_count = sum(1 for s in self.selections if s['correct'])
        
        self.scenario_display.value = f"""
        <div style='border: 3px solid #4CAF50; padding: 30px; border-radius: 15px; background: #f8fff8; text-align: center;'>
            <h2 style='color: #4CAF50; margin: 0 0 20px 0;'>Explanation Technique Selection Complete! 🎯</h2>
            
            <div style='margin: 20px 0;'>
                <h3>Your Results</h3>
                <p style='font-size: 20px;'><strong>Final Score:</strong> {self.score}/{max_score} points ({accuracy:.0f}%)</p>
                <p><strong>Perfect Matches:</strong> {perfect_count}/{total}</p>
            </div>
            
            <h3>Skill Developed: Audience-Appropriate Explanation Selection</h3>
            <ul style='text-align: left; max-width: 700px; margin: 20px auto;'>
                <li><strong>Audience Analysis:</strong> Learned to assess technical background and information needs</li>
                <li><strong>Technique Matching:</strong> Practiced pairing explanation methods with stakeholder contexts</li>
                <li><strong>Trade-off Recognition:</strong> Understood strengths and limitations of each approach</li>
                <li><strong>Context Sensitivity:</strong> Recognized how decision stakes influence explanation needs</li>
            </ul>
            
            <h4>Key Insight</h4>
            <p style='max-width: 700px; margin: 0 auto;'>Effective AI transparency requires matching explanation techniques to audience needs.
            What works for data scientists (SHAP, LIME) often fails for end users, while approaches that build
            public trust (natural language, examples) may lack the rigor needed for regulatory compliance.
            The skill is knowing which technique serves which purpose.</p>
        </div>
        """
        
        self.technique_selector.layout.display = 'none'
        self.technique_info_display.value = ""
        self.submit_button.layout.display = 'none'
        self.feedback_display.value = ""
        self.next_button.layout.display = 'none'

In [ ]:
#@title Start the Module
# Start the Explanation Technique Selector
selector = ExplanationTechniqueSelector(scenarios, explanation_techniques)
widget = selector.start_selector()
display(widget)